<a href="https://colab.research.google.com/github/WVF-1/Movie-Success-Prediction-Model-Comparison/blob/main/Feature%20Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Predicting Movie Success — Notebook 10
## Feature Engineering & Dataset Construction

**Series:** May Newsletter — Movie Intelligence (Capstone)
**Prerequisites:** `movies_clean.parquet`, `directors_clean.parquet`, `audience_clean.parquet`

### Architecture overview
Four parallel Random Forest pipelines, each defining "success" differently, trained on the
same temporal split and evaluated on the same holdout test set.

| Pipeline | Target | Feature layers |
|----------|--------|---------------|
| M1 — ROI lens | ROI > 1× (binary) | Layer 1 only |
| M2 — Revenue lens | Revenue ≥ 60th pct (binary) | Layers 1 + 2 |
| M3 — Ratings lens | avg_user_rating ≥ median (binary) | Layers 1 + 2 |
| M4 — CSI lens | Cinema Success Index ≥ 60th pct (binary) | Layers 1 + 2 + 3 |

### Temporal split
- **Training set:** films released before 2008
- **Test set:** films released 2008 – 2017 (held out until NB12)

### What we do here
1. Merge all three source parquets into one master dataframe
2. Construct the four target variables including the Cinema Success Index (CSI)
3. Engineer Layer 1 (base production features)
4. Engineer the Director Success Score with shrinkage prior (threshold = 5 films)
5. Engineer Layer 2 (creative features) and Layer 3 (audience proxy features)
6. Apply the temporal split and save four clean feature matrices


## 0 · Imports

In [1]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)


pandas : 2.2.2
numpy  : 2.0.2


## 1 · Load & Merge Source Parquets

In [2]:
movies    = pd.read_parquet("movies_clean.parquet")
directors = pd.read_parquet("directors_clean.parquet")
audience  = pd.read_parquet("audience_clean.parquet")

# Harmonise id
for df_ in [movies, directors, audience]:
    df_["id"] = df_["id"].astype(str).str.strip()

# directors_clean already contains all movies_clean columns — use it as base
# and merge the audience behavioural columns on top
AUDIENCE_COLS = ["id","avg_user_rating","rating_std","rating_count",
                 "pct_5star","pct_low","comm_vs_audience_gap","quadrant","polarisation"]
available_audience_cols = [c for c in AUDIENCE_COLS if c in audience.columns]

master = directors.merge(audience[available_audience_cols], on="id", how="left")

print(f"movies_clean    : {movies.shape}")
print(f"directors_clean : {directors.shape}")
print(f"audience_clean  : {audience.shape}")
print(f"master          : {master.shape}")
print(f"  Films with audience data : {master['avg_user_rating'].notna().sum():,}")
print(f"  Release year range       : {int(master['release_year'].min())} – {int(master['release_year'].max())}")


movies_clean    : (5128, 23)
directors_clean : (5140, 27)
audience_clean  : (1657, 35)
master          : (5144, 34)
  Films with audience data : 1,663
  Release year range       : 1915 – 2017


## 2 · Target Variables

In [3]:
# ── All thresholds computed on the FULL dataset for consistent labelling ──────
# (The models only train on pre-2008 rows — thresholds are global population stats)

# T1: success_roi — was the film profitable beyond break-even?
master["success_roi"] = (master["ROI"] > 1.0).astype(int)

# T2: success_revenue — top 40% of earners?
rev_60 = master["revenue"].quantile(0.60)
master["success_revenue"] = (master["revenue"] >= rev_60).astype(int)

# T3: success_rating — did audiences rate it above median?
#     Requires audience data; rows without ratings will be excluded from M3
rat_med = master["avg_user_rating"].median()
master["success_rating"] = (master["avg_user_rating"] >= rat_med).astype(int)

# T4: Cinema Success Index (CSI)
#     CSI = 0.4 × revenue_pct_rank + 0.4 × rating_pct_rank + 0.2 × vote_count_pct_rank
master["rev_pct_rank"]        = master["revenue"].rank(pct=True)
master["rat_pct_rank"]        = master["avg_user_rating"].rank(pct=True, na_option="keep")
master["vote_count_pct_rank"] = master["vote_count"].rank(pct=True)

master["CSI"] = (
    0.4 * master["rev_pct_rank"]
  + 0.4 * master["rat_pct_rank"].fillna(master["rev_pct_rank"])  # fallback for unrated films
  + 0.2 * master["vote_count_pct_rank"]
)

csi_60 = master["CSI"].quantile(0.60)
master["success_CSI"] = (master["CSI"] >= csi_60).astype(int)

print("Target variable distributions:")
for col in ["success_roi","success_revenue","success_rating","success_CSI"]:
    pos = master[col].sum()
    pct = master[col].mean() * 100
    print(f"  {col:<20} : {pos:,} positive  ({pct:.1f}%)")

print(f"\nCSI thresholds:")
print(f"  Revenue 60th pct   : ${rev_60/1e6:.1f}M")
print(f"  Rating median      : {rat_med:.2f}")
print(f"  CSI 60th pct       : {csi_60:.3f}")


Target variable distributions:
  success_roi          : 2,666 positive  (51.8%)
  success_revenue      : 2,058 positive  (40.0%)
  success_rating       : 833 positive  (16.2%)
  success_CSI          : 2,058 positive  (40.0%)

CSI thresholds:
  Revenue 60th pct   : $52.0M
  Rating median      : 3.52
  CSI 60th pct       : 0.563


## 3 · Layer 1 — Base Production Features

In [4]:
# ── 3a: Log-transform budget ──────────────────────────────────────
master["log_budget"] = np.log1p(master["budget"])

# ── 3b: Runtime — impute missing with median ──────────────────────
runtime_median = master["runtime"].median()
master["runtime_clean"] = master["runtime"].fillna(runtime_median)

# ── 3c: Release month — cyclical encoding ─────────────────────────
master["month_sin"] = np.sin(2 * np.pi * master["release_month"] / 12)
master["month_cos"] = np.cos(2 * np.pi * master["release_month"] / 12)

# ── 3d: Language flag ─────────────────────────────────────────────
master["is_english"] = (master["original_language"] == "en").astype(int)

# ── 3e: Genre one-hot (top 10 genres; rest → 'Other') ────────────
TOP_GENRES = master["primary_genre"].value_counts().head(10).index.tolist()
master["genre_clean"] = master["primary_genre"].where(
    master["primary_genre"].isin(TOP_GENRES), other="Other")
genre_dummies = pd.get_dummies(master["genre_clean"], prefix="genre")
master = pd.concat([master, genre_dummies], axis=1)

# ── 3f: Genre count already present ──────────────────────────────
master["genre_count"] = master["genre_count"].fillna(1)

LAYER1_COLS = (
    ["log_budget","runtime_clean","month_sin","month_cos","is_english","genre_count"]
    + list(genre_dummies.columns)
)

print(f"Layer 1 feature count : {len(LAYER1_COLS)}")
print("Features:", LAYER1_COLS)


Layer 1 feature count : 17
Features: ['log_budget', 'runtime_clean', 'month_sin', 'month_cos', 'is_english', 'genre_count', 'genre_Action', 'genre_Adventure', 'genre_Animation', 'genre_Comedy', 'genre_Crime', 'genre_Drama', 'genre_Fantasy', 'genre_Horror', 'genre_Other', 'genre_Romance', 'genre_Thriller']


## 4 · Director Success Score (Layer 2, core feature)

In [5]:
# ── Compute per-director career stats from the TRAINING set only ──────────────
# Threshold = 5 films (consistent with Part 2's minimum for 'powerhouse' status)
SHRINKAGE_THRESHOLD = 5

# We define the training window as pre-2008 for this computation
train_mask = master["release_year"] < 2008
train_df   = master[train_mask & master["director"].notna()].copy()

dir_stats = (
    train_df.groupby("director")
    .agg(
        n_films          = ("title",         "count"),
        dir_roi_mean     = ("ROI",            "mean"),
        dir_rev_mean     = ("revenue",        "mean"),
        dir_rating_mean  = ("avg_user_rating","mean"),
    )
    .reset_index()
)

# Fill missing ratings (directors whose films have no audience data)
dir_stats["dir_rating_mean"] = dir_stats["dir_rating_mean"].fillna(
    dir_stats["dir_rating_mean"].median())

# ── Normalise each component to [0, 1] ───────────────────────────
def norm01(s):
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0

dir_stats["roi_norm"]    = norm01(dir_stats["dir_roi_mean"])
dir_stats["rev_norm"]    = norm01(dir_stats["dir_rev_mean"])
dir_stats["rating_norm"] = norm01(dir_stats["dir_rating_mean"])

# ── Raw weighted score ────────────────────────────────────────────
dir_stats["dir_score_raw"] = (
    0.2 * dir_stats["roi_norm"]
  + 0.3 * dir_stats["rev_norm"]
  + 0.5 * dir_stats["rating_norm"]
)

# ── Shrinkage prior — blend toward population mean for < threshold films ──────
pop_mean = dir_stats["dir_score_raw"].mean()
dir_stats["shrinkage_weight"] = (
    dir_stats["n_films"] / SHRINKAGE_THRESHOLD
).clip(upper=1.0)

dir_stats["director_success_score"] = (
    dir_stats["shrinkage_weight"]       * dir_stats["dir_score_raw"]
  + (1 - dir_stats["shrinkage_weight"]) * pop_mean
)

print(f"Directors with career stats  : {len(dir_stats):,}")
print(f"  ≥ {SHRINKAGE_THRESHOLD} films (full weight)  : {(dir_stats['n_films'] >= SHRINKAGE_THRESHOLD).sum():,}")
print(f"  < {SHRINKAGE_THRESHOLD} films (shrunk)       : {(dir_stats['n_films'] <  SHRINKAGE_THRESHOLD).sum():,}")
print(f"\nScore distribution:")
print(dir_stats["director_success_score"].describe().round(3).to_string())
print(f"\nTop 10 directors by score:")
print(dir_stats.nlargest(10,"director_success_score")
      [["director","n_films","dir_score_raw","director_success_score"]].to_string(index=False))


Directors with career stats  : 1,407
  ≥ 5 films (full weight)  : 160
  < 5 films (shrunk)       : 1,247

Score distribution:
count    1407.000
mean        0.361
std         0.030
min         0.208
25%         0.355
50%         0.356
75%         0.364
max         0.552

Top 10 directors by score:
        director  n_films  dir_score_raw  director_success_score
    George Lucas        6       0.552294                0.552294
   Peter Jackson        7       0.534696                0.534696
   James Cameron        7       0.528950                0.528950
  Andrew Adamson        3       0.600981                0.503963
  Hayao Miyazaki        5       0.485922                0.485922
     Mike Newell        5       0.477679                0.477679
   John Lasseter        4       0.506453                0.476850
Steven Spielberg       24       0.476684                0.476684
   William Wyler        5       0.473346                0.473346
 Charlie Chaplin        6       0.470960            

In [6]:
# ── Merge score back onto master ────────────────────────────────────
master = master.merge(
    dir_stats[["director","director_success_score"]],
    on="director", how="left"
)

# Unknown directors (debut films or missing) → population mean
master["director_success_score"] = master["director_success_score"].fillna(pop_mean)

print(f"director_success_score coverage : {master['director_success_score'].notna().mean()*100:.1f}%")
print(f"  Mean  : {master['director_success_score'].mean():.3f}")
print(f"  Std   : {master['director_success_score'].std():.3f}")


director_success_score coverage : 100.0%
  Mean  : 0.367
  Std   : 0.038


## 5 · Layer 2 — Remaining Creative Features

In [7]:
# ── Cast size (already in directors_clean) ────────────────────────
master["cast_size_clean"] = master["cast_size"].fillna(master["cast_size"].median())

# ── Has known lead — binary flag ─────────────────────────────────
# 'Known' = lead actor whose films average in the top quartile of revenue
top_q_threshold = train_df.groupby("lead_actor")["revenue"].mean().quantile(0.75)
high_value_actors = (
    train_df.groupby("lead_actor")["revenue"].mean()
    .loc[lambda s: s >= top_q_threshold]
    .index.tolist()
)
master["has_known_lead"] = master["lead_actor"].isin(high_value_actors).astype(int)

LAYER2_COLS = ["director_success_score","cast_size_clean","has_known_lead"]

print(f"Layer 2 feature count : {len(LAYER2_COLS)}")
print(f"  Known lead actors identified : {len(high_value_actors):,}")
print(f"  Films with known lead        : {master['has_known_lead'].sum():,}  "
      f"({master['has_known_lead'].mean()*100:.1f}%)")


Layer 2 feature count : 3
  Known lead actors identified : 340
  Films with known lead        : 2,014  (39.2%)


## 6 · Layer 3 — Audience Proxy Features

In [8]:
# vote_average and popularity are available pre-release as TMDB community signals
# (vote_average builds up over the film's lifetime, so treat as a pre-test proxy)
master["vote_average_clean"] = master["vote_average"].fillna(master["vote_average"].median())
master["log_popularity"]     = np.log1p(master["popularity"].fillna(0))

LAYER3_COLS = ["vote_average_clean","log_popularity"]

print(f"Layer 3 feature count : {len(LAYER3_COLS)}")


Layer 3 feature count : 2


## 7 · Temporal Train / Test Split

In [9]:
CUTOFF_YEAR = 2008

train = master[master["release_year"] <  CUTOFF_YEAR].copy()
test  = master[master["release_year"] >= CUTOFF_YEAR].copy()

# Sort both by year (required for TimeSeriesSplit in NB11)
train = train.sort_values("release_year").reset_index(drop=True)
test  = test.sort_values("release_year").reset_index(drop=True)

print(f"Training set  : {len(train):,} films  ({train['release_year'].min():.0f} – {train['release_year'].max():.0f})")
print(f"Test set      : {len(test):,}  films  ({test['release_year'].min():.0f} – {test['release_year'].max():.0f})")
print()
print("Target distributions in training set:")
for col in ["success_roi","success_revenue","success_rating","success_CSI"]:
    pos = train[col].sum()
    pct = train[col].mean() * 100
    print(f"  {col:<20}: {pos:,} positive  ({pct:.1f}%)")
print()
print("Target distributions in test set:")
for col in ["success_roi","success_revenue","success_rating","success_CSI"]:
    pos = test[col].sum()
    pct = test[col].mean() * 100
    print(f"  {col:<20}: {pos:,} positive  ({pct:.1f}%)")


Training set  : 3,151 films  (1915 – 2007)
Test set      : 1,993  films  (2008 – 2017)

Target distributions in training set:
  success_roi         : 1,670 positive  (53.0%)
  success_revenue     : 1,179 positive  (37.4%)
  success_rating      : 708 positive  (22.5%)
  success_CSI         : 1,102 positive  (35.0%)

Target distributions in test set:
  success_roi         : 996 positive  (50.0%)
  success_revenue     : 879 positive  (44.1%)
  success_rating      : 125 positive  (6.3%)
  success_CSI         : 956 positive  (48.0%)


## 8 · Assemble & Save Feature Matrices

In [10]:
# Define feature sets for each model
FEATURES = {
    "M1_ROI"     : LAYER1_COLS,
    "M2_Revenue" : LAYER1_COLS + LAYER2_COLS,
    "M3_Ratings" : LAYER1_COLS + LAYER2_COLS,
    "M4_CSI"     : LAYER1_COLS + LAYER2_COLS + LAYER3_COLS,
}

TARGETS = {
    "M1_ROI"     : "success_roi",
    "M2_Revenue" : "success_revenue",
    "M3_Ratings" : "success_rating",
    "M4_CSI"     : "success_CSI",
}

# M3 and M4 require audience data — drop rows without ratings
def get_subset(df_, model_key):
    if model_key in ("M3_Ratings","M4_CSI"):
        return df_.dropna(subset=["avg_user_rating"]).copy()
    return df_.copy()

import os, pickle

os.makedirs("capstone_data", exist_ok=True)

for key in FEATURES:
    feat_cols = FEATURES[key]
    tgt_col   = TARGETS[key]

    tr = get_subset(train, key)
    te = get_subset(test,  key)

    X_train = tr[feat_cols].fillna(0).values
    y_train = tr[tgt_col].values
    X_test  = te[feat_cols].fillna(0).values
    y_test  = te[tgt_col].values

    meta_train = tr[["title","release_year","primary_genre"]].reset_index(drop=True)
    meta_test  = te[["title","release_year","primary_genre"]].reset_index(drop=True)

    payload = {
        "X_train": X_train, "y_train": y_train,
        "X_test" : X_test,  "y_test" : y_test,
        "feature_names": feat_cols,
        "meta_train": meta_train, "meta_test": meta_test,
    }
    with open(f"capstone_data/{key}.pkl","wb") as f:
        pickle.dump(payload, f)

    print(f"  {key:<12} train={len(X_train):,}  test={len(X_test):,}  "
          f"features={len(feat_cols)}  pos_rate_train={y_train.mean()*100:.1f}%")

print()
print("Saved to capstone_data/ ✔")
print("Proceed to Notebook 11 → Model Training & Validation ▶")


  M1_ROI       train=3,151  test=1,993  features=17  pos_rate_train=53.0%
  M2_Revenue   train=3,151  test=1,993  features=20  pos_rate_train=37.4%
  M3_Ratings   train=1,465  test=198  features=20  pos_rate_train=48.3%
  M4_CSI       train=1,465  test=198  features=22  pos_rate_train=54.1%

Saved to capstone_data/ ✔
Proceed to Notebook 11 → Model Training & Validation ▶
